# Sesión 1 · Presentación, refresco de Python y primer CSV

**Programación avanzada en Python aplicada al análisis social** · miércoles 2 de diciembre de 2026

Objetivos de hoy:

1. Saber qué vamos a hacer en la asignatura y cómo encaja con el resto del máster.
2. **Refrescar** el Python que ya visteis en Fundamentos de programación.
3. Leer un fichero CSV con vuestro propio código y echarle un primer vistazo.

> Ejecuta las celdas con `Shift + Enter`. La primera celda de código prepara la carpeta de trabajo.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

## 1. De qué va esta asignatura

En Fundamentos de programación aprendisteis las piezas de Python: variables, bucles, funciones. Aquí aprenderemos a **usarlas para trabajar con datos de verdad** y, sobre todo, a **organizar el código** para que sea vuestro, se entienda y se pueda reutilizar.

El hilo conductor es este recorrido:

| Etapa | Qué haréis | Sesiones |
| --- | --- | --- |
| Leer | Cargar datos de un CSV y de una API con vuestro propio código | 1, 3 |
| Calcular | Convertir un cálculo en una función con nombre | 2 |
| Ordenar | Unir tablas y guardar vuestras funciones en un fichero propio | 4 |
| Mostrar y explorar | Funciones que dibujan y flujos de pasos encadenados | 5, 6 |
| Compartir y acelerar | Modelos reproducibles, proyecto ordenado y código eficiente | 8, 9, 10 |

No os daremos una caja negra que cargue los datos: **la escribiréis vosotros**, paso a paso.

### Cómo encaja con las demás asignaturas

| Asignatura | Allí aprendéis | Aquí lo usamos para |
| --- | --- | --- |
| Fundamentos de programación | Las bases de Python | Darlas por sabidas (con un refresco hoy) |
| Aprendizaje estadístico | Qué estadístico usar y cómo interpretarlo | Programarlo de forma clara y reutilizable |
| Obtención y preprocesamiento | Bases de datos, *scraping*, métodos de limpieza | Leer CSV y API con pandas; nada de bases de datos |
| Aprendizaje automático | Los algoritmos y su evaluación | Organizar el código de un modelo |

## 2. Refresco exprés de Python

Cinco celdas para recordar lo imprescindible. Si algo no os suena, decidlo ahora; el libro de la asignatura tiene un repaso más pausado.

In [ ]:
# Variables y texto con valores dentro (f-strings)
provincia = "Salamanca"
hipotecas = 212
poblacion = 327_000

tasa = hipotecas / poblacion * 1000
print(f"En {provincia} hubo {tasa:.2f} hipotecas por cada mil habitantes")

In [ ]:
# Listas: una serie de valores. Las posiciones empiezan en cero.
valores = [212, 198, 240, 251]

print(valores[0], valores[-1], len(valores))
print(sum(valores) / len(valores))

In [ ]:
# Diccionarios: pares clave -> valor
poblaciones = {"Salamanca": 327_000, "Granada": 921_000}

print(poblaciones["Granada"])
for nombre, habitantes in poblaciones.items():
    print(nombre, habitantes)

In [ ]:
# Condiciones y bucles
for valor in valores:
    if valor > 220:
        print(valor, "-> por encima de 220")
    else:
        print(valor)

In [ ]:
# Funciones: un cálculo con nombre
def tasa_por_mil(casos, habitantes):
    """Casos por cada mil habitantes."""
    return casos / habitantes * 1000


print(tasa_por_mil(212, 327_000))
print(tasa_por_mil(705, 921_000))

## 3. Un proyecto ordenado: carpetas y rutas

Antes de leer datos hay que saber **dónde están**. Un proyecto de análisis suele tener esta forma:

```
mi_proyecto/
├── datos/          <- los ficheros de datos, sin tocar
├── resultados/     <- tablas y gráficos que genera el código
└── analisis.ipynb  <- el notebook
```

Una **ruta relativa** indica dónde está un fichero *partiendo de la carpeta de trabajo*: `datos/hipotecas.csv` significa «dentro de la carpeta `datos`, el fichero `hipotecas.csv`». Evitad las rutas completas del tipo `C:\Users\ana\Escritorio\...`: solo funcionan en vuestro ordenador.

`pathlib.Path` permite manejar rutas igual en Windows, macOS y Linux.

In [ ]:
from pathlib import Path

carpeta_datos = Path("datos")

print(carpeta_datos.exists())
for fichero in sorted(carpeta_datos.glob("*.csv")):
    print(fichero.name)

> **¿Datos reales o simulados?**
>
>  El fichero `datos/ORIGEN.md` indica de dónde sale cada fichero. Algunos son **simulados**: tienen la misma forma que los reales, pero valores inventados, y sirven para practicar sin conexión. A partir de la sesión 3 descargaréis vosotros los datos reales.

## 4. Leer un CSV con pandas

Un CSV es un fichero de texto con una fila por línea y los valores separados por comas. **pandas** lo convierte en una tabla (un `DataFrame`).

In [ ]:
import pandas as pd

hipotecas = pd.read_csv("datos/hipotecas.csv")
hipotecas.head()

Cada fila es un mes y una provincia. Las columnas son:

- `fecha`: primer día del mes.
- `cod_prov` y `provincia`: código y nombre de la provincia (`00` es el total nacional).
- `hipotecas`: número de hipotecas sobre viviendas firmadas ese mes.
- `importe_miles`: capital prestado, en miles de euros.

### Cuatro preguntas que hacerle siempre a una tabla nueva

In [ ]:
print(hipotecas.shape)          # 1. ¿Cuántas filas y columnas tiene?

In [ ]:
hipotecas.dtypes                # 2. ¿De qué tipo es cada columna?

Dos cosas no han salido bien: `fecha` se ha leído como texto y `cod_prov` como número, así que la provincia `05` se ha convertido en `5`. Se lo indicamos a `read_csv`:

In [ ]:
hipotecas = pd.read_csv(
    "datos/hipotecas.csv",
    dtype={"cod_prov": str},        # los códigos son texto, no números
    parse_dates=["fecha"],          # esta columna es una fecha
)
hipotecas.dtypes

In [ ]:
hipotecas.isna().sum()          # 3. ¿Faltan valores?

In [ ]:
print(hipotecas["fecha"].min(), hipotecas["fecha"].max())     # 4. ¿Qué periodo cubre?
print(hipotecas["provincia"].nunique(), "territorios")

### Quedarse con una parte

Entre corchetes se indica una **condición**, y pandas devuelve solo las filas que la cumplen.

In [ ]:
salamanca = hipotecas[hipotecas["provincia"] == "Salamanca"]
salamanca.tail(3)

In [ ]:
salamanca.plot(x="fecha", y="hipotecas", title="Hipotecas sobre viviendas en Salamanca", legend=False);

### Leer un CSV directamente de internet

`read_csv` también acepta una dirección web. Our World in Data publica cada gráfico como CSV:

In [ ]:
url = "https://ourworldindata.org/grapher/life-expectancy.csv"

try:
    vida = pd.read_csv(url, storage_options={"User-Agent": "Mozilla/5.0"})
    print("Leído de internet")
except Exception as error:
    vida = pd.read_csv("datos/esperanza_vida.csv")
    print("Sin conexión: uso la copia de la carpeta datos")

vida.head(3)

El bloque `try / except` dice: «intenta leerlo de internet y, si falla, usa la copia local». Lo veremos con calma en la sesión 3.

### Guardar un resultado

El resultado de un análisis también se guarda con código, en una carpeta distinta de la de los datos originales.

In [ ]:
Path("resultados").mkdir(exist_ok=True)                      # crea la carpeta si no existe

salamanca.to_csv("resultados/hipotecas_salamanca.csv", index=False)
print(Path("resultados/hipotecas_salamanca.csv").exists())

## 5. Usar datos con responsabilidad

| Fuente | ¿Se puede redistribuir el fichero? | Qué hacemos en el curso |
| --- | --- | --- |
| INE, Banco de España, Our World in Data | Sí, citando la fuente | Lo descargamos con nuestro código |
| CIS, European Social Survey, World Values Survey | **No** (registro y condiciones de uso) | Cada estudiante descarga su copia |
| Portales inmobiliarios y redes sociales | El *scraping* no se usa en esta asignatura | No se usan |

Tres costumbres: citar siempre la fuente, no subir a GitHub microdatos con licencia y no tocar nunca los ficheros originales (se leen, y los resultados se guardan aparte).

## 6. Para la próxima sesión

- Comprueba que puedes abrir los notebooks en Colab con tu cuenta.
- Termina la práctica de hoy.
- Participa en el primer debate del foro.